In [ ]:
import pandas as pd
from ssf.Taxonomy import Taxonomy
from ssf.Constants import *
from collections import defaultdict
from ssf.Configs import load_config

def extract_brace_span(s):
    start = s.find("{{")
    end = s.rfind("}}")
    if start == -1:
        raise ValueError("Opening delimiter '{{' not found.")
    if end == -1:
        raise ValueError("Closing delimiter '}}' not found.")
    if end + 2 <= start:
        raise ValueError("Closing delimiter appears before opening delimiter.")
    return s[start:end + 2]

config = load_config(REPLICATION_CONFIG_PATH)

taxonomy = Taxonomy(taxonomy_dir=TAXONOMY_DIR)

ssf_split_test_df = pd.read_csv(f"{config.dirs.data.corpus}/ssf_split_test.csv")
ssf_split_val_df = pd.read_csv(f"{config.dirs.data.corpus}/ssf_split_val.csv")

ssf_split_test_df = ssf_split_test_df.sample(frac=1, random_state=config.random_seeds.default).reset_index(drop=True)
ssf_split_val_df = ssf_split_val_df.sample(frac=1, random_state=config.random_seeds.default).reset_index(drop=True)

print(ssf_split_val_df['meta.subreddit'])
print(ssf_split_test_df['meta.subreddit'])
df_tups = [('val', ssf_split_val_df), ('test', ssf_split_test_df)]

for split, split_df in df_tups:
  print(split, df.columns)
  for dim in taxonomy.get_dims():
    dim_template_prefix = taxonomy.get_template_prefix(dim=dim)
    dim_responses = defaultdict(list)
    for _, row in split_df.iterrows():
      dim_response = row[f"{config.models.openai_default.replace("-", "_")}_{dim}_gen0"]
      if "{{ERROR}}" in dim_response:
        continue
      dim_responses['id'].append(row['id'])
      dim_responses['response'].append(dim_response.split(dim_template_prefix)[1])

    df = pd.DataFrame(dim_responses)
    # Reorder columns to have "id" first
    cols = ["id"] + [col for col in df.columns if col != "id"]
    df = df[cols]
    df.to_csv(f"{config.dirs.data.annotations}/{split}_{dim}_hum_ann.csv", index=False)
    print(len(df))